<a href="https://colab.research.google.com/github/krupa-16/BIS_LAB/blob/main/Parallelcellularalgorithm_ewastecollectionroutemanagement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import random
import math

# Collection locations: coordinates and priority
locations = {
    'A': (2, 3, 3),
    'B': (5, 4, 2),
    'C': (6, 1, 5),
    'D': (8, 3, 4),
    'E': (3, 7, 1)
}

depot = (0, 0)

# PCA parameters
grid_size = 3
iterations = 100
priority_weight = 0.5

random.seed(42)

# Calculate distance between two locations
def distance(p1, p2):
    return math.sqrt(
        (p1[0] - p2[0])**2 +
        (p1[1] - p2[1])**2
    )

# Calculate route fitness
def fitness(route):
    current = depot
    total_distance = 0
    priority_penalty = 0

    for place in route:
        x, y, priority = locations[place]

        d = distance(current, (x, y))
        total_distance += d
        priority_penalty += priority * (
            total_distance
        )

        current = (x, y)

    # Return to depot
    total_distance += distance(current, depot)

    return total_distance + (
        priority_weight * priority_penalty
    )

# Generate random route
def create_route():
    route = list(locations.keys())
    random.shuffle(route)
    return route

# Find neighbouring cells (Moore neighbourhood)
def get_neighbors(index):
    row = index // grid_size
    col = index % grid_size

    neighbors = []

    for r in range(max(0, row-1),
                   min(grid_size, row+2)):
        for c in range(max(0, col-1),
                       min(grid_size, col+2)):

            n = r * grid_size + c

            if n != index:
                neighbors.append(n)

    return neighbors

# Initialize 9 cells
population = [
    create_route()
    for _ in range(grid_size * grid_size)
]

best_route = min(population, key=fitness)[:]

# Main PCA algorithm
for iteration in range(iterations):

    old_population = [
        route[:] for route in population
    ]

    new_population = []

    for i in range(len(population)):

        neighbors = get_neighbors(i)

        # Find best neighbouring cell
        best_neighbor = min(
            neighbors + [i],
            key=lambda n: fitness(old_population[n])
        )

        # Copy current route
        candidate = old_population[i][:]

        # Update route using neighbour
        target = old_population[best_neighbor]

        place = random.choice(target)
        pos1 = candidate.index(place)
        pos2 = target.index(place)

        candidate[pos1], candidate[pos2] = (
            candidate[pos2], candidate[pos1]
        )

        # Accept better route
        if fitness(candidate) < fitness(old_population[i]):
            new_population.append(candidate)
        else:
            new_population.append(old_population[i])

    population = new_population

    # Update global best
    current_best = min(population, key=fitness)

    if fitness(current_best) < fitness(best_route):
        best_route = current_best[:]

# Display result
print("E-WASTE COLLECTION ROUTE")
print("------------------------")

print("Depot ->", " -> ".join(best_route), "-> Depot")

print("Total Distance:",
      round(sum(
          distance(
              (0, 0) if i == 0 else
              locations[best_route[i-1]][:2],
              locations[place][:2]
          )
          for i, place in enumerate(best_route)
      ) + distance(locations[best_route[-1]][:2], depot), 2))

print("Best Fitness:", round(fitness(best_route), 2))
print("Collection Priority:",
      sum(locations[p][2] for p in best_route))


E-WASTE COLLECTION ROUTE
------------------------
Depot -> A -> C -> D -> B -> E -> Depot
Total Distance: 25.29
Best Fitness: 95.61
Collection Priority: 15
